In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA 1 — CARREGAMENTO DE DADOS
# ════════════════════════════════════════════════════════════════════════
#
# Esta célula carrega o dataset de janelas sísmicas.
# Existem dois modos:
#
#   (A) SUPERVISIONADO — se existir 'dados_otimizacao.npz':
#       Contém X_train_noise (só ruído), X_val (ruído + eventos),
#       y_val (0=ruído, 1=evento). Ideal para otimização com F1.
#
#   (B) NÃO-SUPERVISIONADO — fallback usando 'windows.npz':
#       Todas as janelas vêm de event_waveforms (contêm eventos!).
#       O split 70/30 é feito por ordem sequencial.
#       ⚠️ ATENÇÃO: neste modo, o treino NÃO é "só ruído".
#       Serve para testar reconstrução, mas as métricas de detecção
#       serão enviesadas. Para resultados reais, gere o dataset (A).
#
# IMPORTANTE: o tamanho da janela deve bater com o modelo.
# Se windows.npz tem shape (N, 2400) → o modelo deve aceitar 2400.
# Nunca adapte no teste (split/crop) — corrija na origem.
# ════════════════════════════════════════════════════════════════════════

import numpy as np
import os
from pathlib import Path

# ── Raiz do projeto (ajuste UMA VEZ aqui) ──────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\vish8\OneDrive\Documentos\TCC")

# ── Caminhos derivados ─────────────────────────────────────────────────
windows_dir    = PROJECT_ROOT / "data" / "processed" / "windows"
opt_data_path  = windows_dir / "dados_otimizacao.npz"

alt_dir          = PROJECT_ROOT / "data" / "processed" / "windows_40hz_60s"
alt_windows_path = alt_dir / "windows.npz"
alt_meta_path    = alt_dir / "meta.npy"

print("windows_dir:", windows_dir)
print("opt_data_path:", opt_data_path)
print("alt_windows_path:", alt_windows_path)

meta_val = None

if opt_data_path.exists():
    # ── Modo A: supervisionado ──────────────────────────────────────────
    data = np.load(opt_data_path)
    X_train_noise = data["X_train_noise"]
    X_val = data["X_val"]
    y_val = data["y_val"].ravel().astype(int)
    print("✅ Modo: SUPERVISIONADO (dados_otimizacao.npz)")
else:
    # ── Modo B: não-supervisionado (fallback) ───────────────────────────
    if not alt_windows_path.exists():
        raise FileNotFoundError(
            "Não encontrei dados. Esperava 'dados_otimizacao.npz' em "
            f"{windows_dir} ou 'windows.npz' em {alt_windows_path.parent}."
        )
    X_all = np.load(alt_windows_path)["X"]
    meta_all = (
        np.load(alt_meta_path, allow_pickle=True)
        if alt_meta_path.exists()
        else None
    )
    n = X_all.shape[0]
    split = int(0.7 * n)
    X_train_noise = X_all[:split]
    X_val = X_all[split:]
    y_val = None
    meta_val = meta_all[split:] if meta_all is not None else None
    print("⚠️ Modo: NÃO-SUPERVISIONADO (windows.npz + meta.npy)")
    print(f"   Split: train_ref={split} | val={n - split}")

# ── Resumo dos dados carregados ────────────────────────────────────────
INPUT_DIM = X_train_noise.shape[1]  # dimensão que o modelo precisa aceitar
print(f"\nDados carregados:")
print(f"  X_train_noise: {X_train_noise.shape}  ({INPUT_DIM} amostras = {INPUT_DIM/40:.0f}s @ 40Hz)")
print(f"  X_val:         {X_val.shape}")
print(f"  y_val:         {None if y_val is None else (y_val.shape, np.unique(y_val))}")

windows_dir: C:\Users\vish8\OneDrive\Documentos\TCC\data\processed\windows
opt_data_path: C:\Users\vish8\OneDrive\Documentos\TCC\data\processed\windows\dados_otimizacao.npz
alt_windows_path: C:\Users\vish8\OneDrive\Documentos\TCC\data\processed\windows_40hz_60s\windows.npz
Modo: não-supervisionado (windows.npz + meta.npy)
Split: train_ref=4854 | val=2081
Dados carregados:
X_train_noise: (4854, 2400)
X_val: (2081, 2400)
y_val: None


In [2]:
!pip install tensorflow

In [11]:
!pip install optuna numpy scikit-learn matplotlib

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA 4 — IMPORTS PRINCIPAIS
# ════════════════════════════════════════════════════════════════════════
#
# Importamos tudo que precisamos para:
#   - Construir o autoencoder (TensorFlow/Keras)
#   - Otimizar hiperparâmetros (Optuna com TPE sampler)
#   - Avaliar desempenho (sklearn F1, precision, recall)
#   - Callbacks: EarlyStopping e ReduceLROnPlateau
#
# NOTA sobre EarlyStopping:
#   Sem ela, o modelo treina N épocas fixas e pode:
#   (a) parar cedo demais (subfit) ou
#   (b) memorizar o treino (overfit).
#   Com EarlyStopping(monitor="val_loss", patience=5), o treino para
#   automaticamente quando a loss de validação para de melhorar há 5 épocas.
#
# NOTA sobre ReduceLROnPlateau:
#   Se a val_loss estagnar, reduz o learning rate pela metade.
#   Isso permite "afinar" o modelo sem explodir o gradiente.
# ════════════════════════════════════════════════════════════════════════

import os
os.environ['MPLBACKEND'] = 'agg'     # evita pop-up de matplotlib no notebook
import matplotlib
matplotlib.use('Agg')

import optuna
from optuna.samplers import TPESampler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import f1_score
import numpy as np

print(f"TensorFlow: {tf.__version__}")
print(f"Optuna: {optuna.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

c:\Users\vish8\OneDrive\Documentos\TCC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA 5 — FUNÇÃO OBJETIVO DO OPTUNA (objective)
# ════════════════════════════════════════════════════════════════════════
#
# O Optuna chama esta função N vezes (uma por trial). Em cada trial:
#   1) Sugere hiperparâmetros (n_camadas, latent_dim, dropout, lr, batch)
#   2) Constrói um autoencoder MLP com esses parâmetros
#   3) Treina com EarlyStopping (máx 50 épocas, para se val_loss estagnar)
#   4) Calcula o erro de reconstrução no treino → define threshold
#   5) Aplica threshold no val → predição binária → calcula F1
#   6) Retorna F1 (Optuna tenta MAXIMIZAR)
#
# ESTRUTURA DO AUTOENCODER (MLP simétrico):
#   Input(INPUT_DIM) → Dense(256) → [Dense(128)]×(n-2) → Dense(latent_dim)
#                     ↓  encoder                          bottleneck
#   Dense(latent_dim) → [Dense(128)]×(n-2) → Dense(256) → Dense(INPUT_DIM)
#                     ↓  decoder                          output (linear)
#
# Por que INPUT_DIM e não 1200?
#   Porque agora usamos a dimensão real dos dados (2400 se 60s, 1200 se 30s).
#   Não há mais "adaptação" — o modelo bate com o dado na origem.
#
# NOTA: Esta função SÓ funciona no modo supervisionado (y_val existe).
#   No modo não-supervisionado, não temos rótulo → não dá pra calcular F1.
# ════════════════════════════════════════════════════════════════════════

def build_autoencoder(input_dim, n_camadas, latent_dim, dropout_rate, learning_rate):
    """
    Constrói um autoencoder MLP (Dense) simétrico.

    O encoder comprime input_dim → latent_dim.
    O decoder reconstrói latent_dim → input_dim.
    A loss é MSE (Mean Squared Error) entre entrada e saída.
    """
    model = Sequential(name="autoencoder")

    # ── Encoder ─────────────────────────────────────────────────────────
    # Primeira camada: reduz de input_dim para 256 neurônios
    model.add(Dense(256, activation='relu', input_shape=(input_dim,)))
    model.add(Dropout(dropout_rate))

    # Camadas intermediárias do encoder (opcionais, depende de n_camadas)
    for _ in range(n_camadas - 2):
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(dropout_rate))

    # Bottleneck (gargalo): a representação comprimida
    # Aqui o modelo é FORÇADO a aprender apenas as features mais relevantes.
    # Quanto menor latent_dim, mais compressão → mais seletivo.
    model.add(Dense(latent_dim, activation='relu', name='bottleneck'))

    # ── Decoder (simétrico ao encoder) ──────────────────────────────────
    for _ in range(n_camadas - 2):
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(dropout_rate))

    model.add(Dense(256, activation='relu'))

    # Saída linear: reconstrução do sinal original
    # (linear porque os dados são z-score, podem ser negativos)
    model.add(Dense(input_dim, activation='linear'))

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def objective(trial):
    """Função objetivo para o Optuna. Retorna F1-score."""

    # ── Hiperparâmetros sugeridos pelo Optuna ───────────────────────────
    n_camadas    = trial.suggest_int('n_camadas', 2, 5)
    latent_dim   = trial.suggest_int('latent_dim', 16, 128, step=16)
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5)
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [32, 64, 128])

    # ── Construir modelo ────────────────────────────────────────────────
    model = build_autoencoder(INPUT_DIM, n_camadas, latent_dim, dropout_rate, learning_rate)

    # ── Callbacks ───────────────────────────────────────────────────────
    # EarlyStopping: para o treino se val_loss não melhorar em 5 épocas.
    # restore_best_weights: volta ao melhor modelo, não ao último.
    early_stop = EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=0
    )
    # ReduceLROnPlateau: se val_loss estagnar por 3 épocas, lr *= 0.5
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=0
    )

    # ── Treinamento ─────────────────────────────────────────────────────
    # O autoencoder aprende a reconstruir X_train_noise → X_train_noise.
    # Se treinado SÓ com ruído, ele fica "bom" em reconstruir ruído,
    # e "ruim" em reconstruir eventos → erro alto = anomalia detectada.
    model.fit(
        X_train_noise, X_train_noise,
        epochs=50,                    # máximo (early stop geralmente para antes)
        batch_size=batch_size,
        validation_split=0.1,         # 10% do treino para monitorar overfitting
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )

    # ── Threshold via erro de reconstrução no treino ────────────────────
    # Calcula MSE por janela no conjunto de treino.
    # O percentil p define "a partir de qual erro consideramos anômalo".
    reconst_train = model.predict(X_train_noise, verbose=0)
    errors_train = np.mean((X_train_noise - reconst_train)**2, axis=1)

    thr_percentile = trial.suggest_int('thr_percentile', 90, 99)
    thr = np.percentile(errors_train, thr_percentile)

    # ── Avaliação no validation ─────────────────────────────────────────
    reconst_val = model.predict(X_val, verbose=0)
    errors_val = np.mean((X_val - reconst_val)**2, axis=1)
    y_pred = (errors_val > thr).astype(int)

    # F1-score: média harmônica de precision e recall.
    # É a métrica ideal quando as classes são desbalanceadas (poucos eventos).
    f1 = f1_score(y_val, y_pred)

    return f1

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA 6 — EXECUTAR OTIMIZAÇÃO COM OPTUNA
# ════════════════════════════════════════════════════════════════════════
#
# TPESampler (Tree-structured Parzen Estimator):
#   Sampler bayesiano que aprende quais regiões do espaço de
#   hiperparâmetros são promissoras e foca nelas.
#   Muito mais eficiente que grid search ou random search.
#
# n_trials: quantas combinações testar. Comece com 30, depois aumente.
#   Cada trial treina um modelo completo, então demora.
#
# direction='maximize': queremos o MAIOR F1-score possível.
#
# ⚠️ Esta célula só funciona no modo supervisionado (y_val != None).
#   No modo não-supervisionado, pule direto para a célula de treino
#   manual (sem Optuna).
# ════════════════════════════════════════════════════════════════════════

if y_val is None:
    print("⚠️ Modo não-supervisionado: Optuna desabilitado (sem y_val para calcular F1).")
    print("   Pule para a célula de treino manual mais abaixo.")
    study = None
else:
    sampler = TPESampler(seed=42)   # seed fixa → resultados reprodutíveis
    study = optuna.create_study(direction='maximize', sampler=sampler)

    study.optimize(objective, n_trials=30, show_progress_bar=True)

    print("\n═══ Melhores parâmetros encontrados ═══")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    print(f"\n  Melhor F1-score: {study.best_value:.4f}")

[I 2026-03-03 18:10:40,291] A new study created in memory with name: no-name-13c5c87f-bd49-4cdd-9c59-7c13ecafa89e
  0%|          | 0/30 [00:00<?, ?it/s]C:\Users\vish8\AppData\Local\Temp\ipykernel_6556\3675829207.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
c:\Users\vish8\OneDrive\Documentos\TCC\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1008/1008 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


Best trial: 0. Best value: 0.0131291:   3%|▎         | 1/30 [02:38<1:16:27, 158.19s/it]

[I 2026-03-03 18:13:18,488] Trial 0 finished with value: 0.01312910284463895 and parameters: {'n_camadas': 3, 'latent_dim': 128, 'dropout_rate': 0.36599697090570255, 'lr': 0.0015751320499779737, 'batch_size': 16, 'thr_percentile': 98}. Best is trial 0 with value: 0.01312910284463895.


C:\Users\vish8\AppData\Local\Temp\ipykernel_6556\3675829207.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
c:\Users\vish8\OneDrive\Documentos\TCC\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

[I 2026-03-03 19:07:00,442] Trial 28 finished with value: 0.006048387096774193 and parameters: {'n_camadas': 5, 'latent_dim': 64, 'dropout_rate': 0.3873472461020892, 'lr': 0.0003175633061448995, 'batch_size': 64, 'thr_percentile': 95}. Best is trial 6 with value: 0.023529411764705882.


C:\Users\vish8\AppData\Local\Temp\ipykernel_6556\3675829207.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-4, 1e-2)
c:\Users\vish8\OneDrive\Documentos\TCC\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1008/1008 ━━━━━━━━━━━━━━━━━━━━ 1s 929us/step
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 994us/step


Best trial: 6. Best value: 0.0235294: 100%|██████████| 30/30 [57:37<00:00, 115.24s/it]

[I 2026-03-03 19:08:17,612] Trial 29 finished with value: 0.01312910284463895 and parameters: {'n_camadas': 3, 'latent_dim': 16, 'dropout_rate': 0.41965754840705194, 'lr': 0.002854138917098471, 'batch_size': 32, 'thr_percentile': 98}. Best is trial 6 with value: 0.023529411764705882.
Melhores parâmetros encontrados:
{'n_camadas': 4, 'latent_dim': 32, 'dropout_rate': 0.4847923138822793, 'lr': 0.0035503048581283078, 'batch_size': 16, 'thr_percentile': 99}
Melhor F1-score: 0.0235


In [12]:
!pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl (236 kB)

   ---------------------------------------- 0/7 [fastjsonschema]
   ----- ---------------

In [14]:
import nbformat
import sys
print("nbformat:", nbformat.__version__)
print("python:", sys.executable)

nbformat: 5.10.4
python: c:\Users\vish8\OneDrive\Documentos\TCC\.venv\Scripts\python.exe


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA — VISUALIZAÇÕES DO OPTUNA
# ════════════════════════════════════════════════════════════════════════
#
# plot_optimization_history: mostra como o F1 evolui trial a trial.
#   Se a curva sobe e estabiliza, a busca convergiu.
#   Se ainda sobe no final, vale rodar mais trials.
#
# plot_param_importances: mostra quais hiperparâmetros mais impactam F1.
#   Exemplo: se "lr" domina, os outros parâmetros importam menos.
# ════════════════════════════════════════════════════════════════════════

if study is not None:
    import optuna.visualization as vis

    fig1 = vis.plot_optimization_history(study)
    fig1.show()

    fig2 = vis.plot_param_importances(study)
    fig2.show()
else:
    print("Optuna não foi executado (modo não-supervisionado).")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA — SALVAR ESTUDO OPTUNA (para retomar depois)
# ════════════════════════════════════════════════════════════════════════
#
# O objeto `study` contém TODOS os trials (parâmetros, métricas, etc.).
# Salvando com joblib, você pode:
#   - Carregar depois e ver os resultados sem re-treinar
#   - Continuar a otimização adicionando mais trials
#   - Gerar gráficos adicionais
# ════════════════════════════════════════════════════════════════════════

if study is not None:
    import joblib
    study_path = windows_dir / "optuna_study.pkl"
    study_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(study, str(study_path))
    print("Estudo salvo em:", study_path)
else:
    print("Nada para salvar (Optuna não foi executado).")

Estudo salvo em: C:\Users\vish8\OneDrive\Documentos\SeriesTemporaisSismicas\data\processed\windows\optuna_study.pkl


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA — TREINAR MODELO FINAL
# ════════════════════════════════════════════════════════════════════════
#
# Duas situações possíveis:
#
# (A) Optuna rodou → usa study.best_params
# (B) Sem Optuna   → usa parâmetros DEFAULT razoáveis
#
# Em ambos os casos:
#   - Treina com EarlyStopping (máx 100 épocas, patience=10)
#   - Salva o melhor modelo (.h5)
#   - INPUT_DIM vem direto dos dados (sem adaptação)
#
# SOBRE ÉPOCAS:
#   No treino final, usamos mais épocas (100) porque não estamos
#   buscando hiperparâmetros — queremos extrair o máximo do modelo.
#   O EarlyStopping garante que não vai overfitar.
# ════════════════════════════════════════════════════════════════════════

if study is not None and study.best_trial is not None:
    best = study.best_params
    print("Usando parâmetros do Optuna:")
else:
    # Parâmetros default razoáveis (para modo não-supervisionado)
    best = {
        'n_camadas': 3,
        'latent_dim': 64,
        'dropout_rate': 0.2,
        'lr': 1e-3,
        'batch_size': 64,
    }
    print("⚠️ Usando parâmetros DEFAULT (Optuna não rodou):")

for k, v in best.items():
    print(f"  {k}: {v}")

# ── Construir modelo ────────────────────────────────────────────────────
model_final = build_autoencoder(
    input_dim=INPUT_DIM,
    n_camadas=best['n_camadas'],
    latent_dim=best['latent_dim'],
    dropout_rate=best['dropout_rate'],
    learning_rate=best['lr'],
)
model_final.summary()

# ── Callbacks ───────────────────────────────────────────────────────────
callbacks_final = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,                 # mais paciente no treino final
        restore_best_weights=True,   # recupera o melhor checkpoint
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]

# ── Treinamento ─────────────────────────────────────────────────────────
history_final = model_final.fit(
    X_train_noise, X_train_noise,   # autoencoder: entrada = saída esperada
    epochs=100,                      # máximo; early stop para antes
    batch_size=best['batch_size'],
    validation_split=0.1,
    callbacks=callbacks_final,
    verbose=1,
)

# ── Salvar modelo ───────────────────────────────────────────────────────
windows_dir.mkdir(parents=True, exist_ok=True)
model_path = windows_dir / "autoencoder_final.h5"
model_final.save(str(model_path))
print(f"\n✅ Modelo final salvo em: {model_path}")
print(f"   Input dim: {INPUT_DIM}  ({INPUT_DIM/40:.0f}s @ 40Hz)")

c:\Users\vish8\OneDrive\Documentos\TCC\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0247 - val_loss: 0.0289
Epoch 2/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0247 - val_loss: 0.0289
Epoch 3/50
1815/1815 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0247 - val_loss: 0.0290
Epoch 4/50
1246/1815 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0244

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA — DISTRIBUIÇÃO DOS ERROS DE RECONSTRUÇÃO
# ════════════════════════════════════════════════════════════════════════
#
# Após treinar o modelo final, calculamos o MSE (Mean Squared Error) por
# janela no conjunto de VALIDAÇÃO. As janelas que o modelo reconstrói
# mal (erro alto) são candidatas a "evento sísmico".
#
# Dois cenários:
#   (A) Supervisionado: temos y_val → histograma separado por classe
#   (B) Não-supervisionado: sem y_val → histograma único de todos os erros
#
# INSIGHTS VISUAIS:
#   - Se as curvas de "Ruído" e "Evento" se separam bem → bom sinal!
#   - Se se sobrepõem muito → threshold vai ter muitos falsos positivos.
#   - O percentil 99 do ruído é um bom ponto de partida para threshold.
# ════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go

# ── Calcular erros no validation ────────────────────────────────────────
reconst_val = model_final.predict(X_val)
errors_val = np.mean((X_val - reconst_val)**2, axis=1)

fig = go.Figure()

if y_val is not None:
    # ── Modo supervisionado: separar por classe ─────────────────────────
    erros_ruido  = errors_val[y_val == 0]
    erros_evento = errors_val[y_val == 1]

    fig.add_trace(go.Histogram(
        x=erros_ruido,
        nbinsx=50,
        histnorm='probability density',
        name='Ruído (y=0)',
        opacity=0.6,
        marker_color='#636EFA',       # azul
    ))
    fig.add_trace(go.Histogram(
        x=erros_evento,
        nbinsx=10,
        histnorm='probability density',
        name='Evento (y=1)',
        opacity=0.6,
        marker_color='#EF553B',       # vermelho
    ))

    fig.update_layout(
        title='Distribuição dos Erros de Reconstrução por Classe',
        xaxis_title='Erro de reconstrução (MSE)',
        yaxis_title='Densidade',
        barmode='overlay',
    )
    fig.show()

    # Estatísticas descritivas
    print(f"Erro médio  — ruído:  {np.mean(erros_ruido):.6f}")
    print(f"Erro médio  — evento: {np.mean(erros_evento):.6f}")
    print(f"Erro mediano — ruído:  {np.median(erros_ruido):.6f}")
    print(f"Erro mediano — evento: {np.median(erros_evento):.6f}")
    print(f"Percentil 95 ruído: {np.percentile(erros_ruido, 95):.6f}")
    print(f"Percentil 99 ruído: {np.percentile(erros_ruido, 99):.6f}")

else:
    # ── Modo não-supervisionado: histograma único ───────────────────────
    fig.add_trace(go.Histogram(
        x=errors_val,
        nbinsx=80,
        name='Todos os erros',
        marker_color='#636EFA',
    ))
    fig.update_layout(
        title='Distribuição dos Erros de Reconstrução (sem rótulos)',
        xaxis_title='Erro de reconstrução (MSE)',
        yaxis_title='Contagem',
    )
    fig.show()

    print(f"Erro médio:   {np.mean(errors_val):.6f}")
    print(f"Erro mediano: {np.median(errors_val):.6f}")
    print(f"Percentil 95: {np.percentile(errors_val, 95):.6f}")
    print(f"Percentil 99: {np.percentile(errors_val, 99):.6f}")

253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


AttributeError: module 'matplotlib' has no attribute 'backends'

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA — TESTAR O MODELO SALVO (.h5)
# ════════════════════════════════════════════════════════════════════════
#
# OBJETIVO:
#   Carregar o modelo salvo do disco e avaliar no conjunto de validação.
#   Isso simula o que aconteceria em "produção" (ex: no Raspberry Pi).
#
# FLUXO:
#   1. Carregar modelo com compile=False (não precisa do otimizador)
#   2. Verificar que a dimensão bate (INPUT_DIM == model.input_shape)
#   3. Calcular erros de reconstrução no treino → threshold
#   4. Calcular erros no validation → predição binária
#   5. Se houver rótulos → métricas supervisionadas (F1, AUC, etc.)
#      Se não → listar as janelas mais anômalas
#
# SOBRE O THRESHOLD:
#   Definimos o threshold usando o percentil P dos erros do TREINO.
#   Exemplo: P99 → "só 1% das janelas de RUÍDO teriam erro acima disso".
#   Quanto maior P → menos falsos positivos, mas pode perder eventos.
#   Testar vários valores: 90, 95, 97, 99.
#
# ⚠️  NOTA: não existe mais a função adapt_input_dim!
#   Agora, o modelo e os dados têm a MESMA dimensão (INPUT_DIM = 2400).
#   Se você está carregando um modelo antigo com input_dim=1200,
#   RETREINE com os novos dados de 2400 amostras!
# ════════════════════════════════════════════════════════════════════════

from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

# ── 1. Carregar modelo ──────────────────────────────────────────────────
model_path = Path(windows_dir) / "autoencoder_final.h5"
print(f"Carregando modelo: {model_path}")

ae = tf.keras.models.load_model(str(model_path), compile=False)

expected_dim = ae.input_shape[-1]
print(f"Dimensão esperada pelo modelo: {expected_dim}")
print(f"Dimensão dos dados (INPUT_DIM): {INPUT_DIM}")

# Checar compatibilidade — se não bater, PARE e retreine!
assert expected_dim == INPUT_DIM, (
    f"ERRO: modelo espera {expected_dim} features, mas os dados têm {INPUT_DIM}.\n"
    f"Retreine o modelo com os novos dados de {INPUT_DIM} amostras!"
)

# ── 2. Erros no TREINO → definir threshold ──────────────────────────────
#
# O threshold é construído sobre os erros do TREINO (só ruído).
# Se o autoencoder aprende a reconstruir ruído, um EVENTO terá erro alto.

reconst_train = ae.predict(X_train_noise, verbose=0)
errors_train = np.mean((X_train_noise - reconst_train) ** 2, axis=1)

# Testar vários percentis para ver o impacto no F1
THR_PERCENTILE = 99   # ← Ajuste aqui: 90, 95, 97, 99
thr = np.percentile(errors_train, THR_PERCENTILE)
print(f"\nThreshold (percentil {THR_PERCENTILE}): {thr:.6g}")

# ── 3. Erros no VALIDATION → predição ───────────────────────────────────
reconst_val = ae.predict(X_val, verbose=0)
errors_val = np.mean((X_val - reconst_val) ** 2, axis=1)

# Predição binária: se erro > threshold → classificado como "evento"
y_score = errors_val                           # score contínuo (para AUC)
y_pred  = (y_score > thr).astype(int)          # binário (para F1)

n_acima = int(y_pred.sum())
print(f"Janelas acima do threshold: {n_acima} / {len(y_pred)} "
      f"({100.0 * y_pred.mean():.2f}%)")

# ── 4. Métricas ─────────────────────────────────────────────────────────
results = {
    "thr_percentile": int(THR_PERCENTILE),
    "thr": float(thr),
    "input_dim": int(INPUT_DIM),
}

if y_val is not None:
    # ▸ Modo SUPERVISIONADO — métricas completas
    cm = confusion_matrix(y_val, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_val, y_pred, average="binary", zero_division=0,
    )
    roc_auc = roc_auc_score(y_val, y_score)
    pr_auc  = average_precision_score(y_val, y_score)

    print("\n── Métricas Supervisionadas ──")
    print(f"Confusion matrix [[TN FP],[FN TP]]:\n{cm}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")

    results.update({
        "confusion_matrix": cm,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
    })
else:
    # ▸ Modo NÃO-SUPERVISIONADO — listar as janelas mais anômalas
    print("\n── Top Anomalias (sem rótulos) ──")
    top_k = 10
    top_idx = np.argsort(-y_score)[:top_k]
    for rank, idx in enumerate(top_idx, start=1):
        score = float(y_score[idx])
        print(f"  #{rank:02d}  idx={idx}  MSE={score:.6g}")

# ── 5. Salvar resultados em .npz ────────────────────────────────────────
out_path = Path(windows_dir) / f"teste_autoencoder_p{THR_PERCENTILE}.npz"

save_dict = {
    "errors_val": errors_val,
    "y_pred": y_pred,
    "y_score": y_score,
    "thr": np.array(thr),
    "thr_percentile": np.array(THR_PERCENTILE),
}
if y_val is not None:
    save_dict["y_val"]            = y_val
    save_dict["confusion_matrix"] = results["confusion_matrix"]

np.savez(out_path, **save_dict)
print(f"\n✅ Resultados salvos em: {out_path}")

Carregando modelo: C:\Users\vish8\OneDrive\Documentos\TCC\data\processed\windows\autoencoder_final.h5
Dimensão esperada pelo modelo: 1200
Shapes: X_train (4854, 2400) -> (9708, 1200) (mode=split2)
Shapes: X_val   (2081, 2400) -> (4162, 1200) (mode=split2)
map_val segmentos (seg->count): {0: 2081, 1: 2081}
Threshold (p99): 1.87375
Val acima do threshold: 67 / 4162 (1.61%)
Top 10 maiores erros (índices no X_val_used): [1437, 1441, 1402, 1400, 1401, 1399, 755, 757, 760, 1436]
#01 idx=1437 score=2.37505 orig=1437 seg=0 meta={'source_file': '37509312.ms', 'station': 'CWC', 'channel': 'BHN', 'starttime': '2016-01-01T06:46:06.794536Z', 'endtime': '2016-01-01T06:47:20.769536Z'}
#02 idx=1441 score=2.36671 orig=1441 seg=0 meta={'source_file': '37509312.ms', 'station': 'CWC', 'channel': 'HNZ', 'starttime': '2016-01-01T06:46:05.178391Z', 'endtime': '2016-01-01T06:47:19.098391Z'}
#03 idx=1402 score=2.34126 orig=1402 seg=0 meta={'source_file': '37509312.ms', 'station': 'CGO', 'channel': 'HNZ', 'star

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CÉLULA — CURVAS ROC E PRECISION-RECALL
# ════════════════════════════════════════════════════════════════════════
#
# Estes gráficos são ESSENCIAIS para o TCC e para avaliar a qualidade
# do detector de eventos.
#
# ROC (Receiver Operating Characteristic):
#   - Eixo X: FPR (False Positive Rate) — fração de ruído classificada
#     incorretamente como evento.
#   - Eixo Y: TPR (True Positive Rate / Recall) — fração de eventos
#     detectados corretamente.
#   - Quanto mais perto do canto superior esquerdo, melhor.
#   - AUC = 0.5 → modelo aleatório; AUC = 1.0 → perfeito.
#
# Precision-Recall (PR):
#   - Mais informativa quando as classes são DESBALANCEADAS (poucos
#     eventos, muito ruído) — que é exatamente nosso cenário.
#   - High Precision + High Recall = modelo confiável.
#   - AP (Average Precision) resume a curva num único número.
#
# Se não temos rótulos (modo não-supervisionado):
#   → mostramos o histograma de erros com a linha do threshold.
# ════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go

if y_val is not None:
    from sklearn.metrics import roc_curve, precision_recall_curve

    # ── Curva ROC ───────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_val, y_score)

    fig_roc = go.Figure()
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines", name="ROC",
        line=dict(color='#636EFA', width=2),
    ))
    # Linha diagonal = classificador aleatório (baseline)
    fig_roc.add_shape(
        type="line", x0=0, y0=0, x1=1, y1=1,
        line=dict(dash="dash", color="gray"),
    )
    fig_roc.update_layout(
        title=f"Curva ROC (AUC = {results.get('roc_auc', float('nan')):.4f})",
        xaxis_title="False Positive Rate (FPR)",
        yaxis_title="True Positive Rate (Recall)",
        width=700, height=500,
    )
    fig_roc.show()

    # ── Curva Precision-Recall ──────────────────────────────────────────
    prec, rec, _ = precision_recall_curve(y_val, y_score)

    fig_pr = go.Figure()
    fig_pr.add_trace(go.Scatter(
        x=rec, y=prec, mode="lines", name="PR",
        line=dict(color='#EF553B', width=2),
    ))
    fig_pr.update_layout(
        title=f"Curva Precision-Recall (AP = {results.get('pr_auc', float('nan')):.4f})",
        xaxis_title="Recall",
        yaxis_title="Precision",
        width=700, height=500,
    )
    fig_pr.show()

    print(f"ROC-AUC: {results.get('roc_auc', 'N/A'):.4f}")
    print(f"PR-AUC:  {results.get('pr_auc', 'N/A'):.4f}")

else:
    # ── Sem rótulos → histograma de erros com threshold ─────────────────
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=y_score, nbinsx=80,
        name="Erro de reconstrução",
        marker_color='#636EFA',
    ))
    # Linha vertical no threshold
    fig.add_vline(
        x=results["thr"],
        line_dash="dash",
        line_color="red",
        annotation_text=f"Threshold (P{results['thr_percentile']})",
        annotation_position="top right",
    )
    fig.update_layout(
        title="Distribuição do Erro de Reconstrução — Validação",
        xaxis_title="Erro (MSE)",
        yaxis_title="Contagem",
        width=800, height=500,
    )
    fig.show()

    print(f"Threshold (P{results['thr_percentile']}): {results['thr']:.6g}")
    print(f"Janelas acima: {int(y_pred.sum())} / {len(y_pred)}")